# Tape Preview — Grayscale Threshold

Color channel has lens-shading artifacts on this camera.  
Using **grayscale intensity thresholding** — detect tape by brightness, not color.

View: **frame + detection** (left) + **thresholded mask** (right).

In [ ]:
import time
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

def gstreamer_pipeline(width=640, height=480, fps=30):
    return (
        f"nvarguscamerasrc ! "
        f"video/x-raw(memory:NVMM), width={width}, height={height}, "
        f"format=NV12, framerate={fps}/1 ! "
        f"nvvidconv flip-method=0 ! "
        f"video/x-raw, width={width}, height={height}, format=BGRx ! "
        f"videoconvert ! video/x-raw, format=BGR ! appsink"
    )

cap = cv2.VideoCapture(gstreamer_pipeline(), cv2.CAP_GSTREAMER)
assert cap.isOpened(), "Camera failed to open!"

image_widget = widgets.Image(format='jpeg', width=1280, height=480)

wide = widgets.Layout(width='500px')
style = {'description_width': '100px'}
t_lo     = widgets.IntSlider(value=0,   min=0, max=255, description='gray lo',  layout=wide, style=style, continuous_update=True)
t_hi     = widgets.IntSlider(value=90,  min=0, max=255, description='gray hi',  layout=wide, style=style, continuous_update=True)
blur     = widgets.IntSlider(value=5,   min=1, max=21,  step=2,  description='blur k',   layout=wide, style=style, continuous_update=True)
min_area = widgets.IntSlider(value=300, min=50, max=5000, step=50, description='min area', layout=wide, style=style, continuous_update=True)
live = widgets.Label(value='live values: --')

display(image_widget)
display(widgets.VBox([t_lo, t_hi, blur, min_area, live]))

def bgr_to_jpeg(frame):
    _, buf = cv2.imencode('.jpeg', frame)
    return bytes(buf)

print("Camera ready.")

In [ ]:
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        h, w = frame.shape[:2]
        center_x = w // 2
        bot_pt = (center_x, h - 1)

        # Read slider values ONCE per frame so they show up in live label
        lo_v  = t_lo.value
        hi_v  = t_hi.value
        k_v   = blur.value if blur.value % 2 == 1 else blur.value + 1
        min_v = min_area.value
        live.value = f'live values: lo={lo_v} hi={hi_v} blur={k_v} min_area={min_v}'

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (k_v, k_v), 0)

        mask = cv2.inRange(gray, lo_v, hi_v)
        mask = cv2.erode(mask,  None, iterations=2)
        mask = cv2.dilate(mask, None, iterations=2)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        vis = frame.copy()
        cv2.line(vis, (center_x, 0), (center_x, h), (0, 255, 255), 1)

        detections = []
        for c in contours:
            if cv2.contourArea(c) < min_v:
                continue
            M = cv2.moments(c)
            if M['m00'] == 0:
                continue
            cx = int(M['m10'] / M['m00'])
            cy = int(M['m01'] / M['m00'])
            detections.append((cx, cy, c))

        left_blobs  = [d for d in detections if d[0] <  center_x]
        right_blobs = [d for d in detections if d[0] >= center_x]

        for cx, cy, c in left_blobs:
            cv2.drawContours(vis, [c], -1, (255, 0, 0), 2)
            cv2.circle(vis, (cx, cy), 6, (255, 0, 0), -1)
        for cx, cy, c in right_blobs:
            cv2.drawContours(vis, [c], -1, (0, 128, 255), 2)
            cv2.circle(vis, (cx, cy), 6, (0, 128, 255), -1)

        left_boundary  = min(left_blobs,  key=lambda d: d[0]) if left_blobs  else None
        right_boundary = max(right_blobs, key=lambda d: d[0]) if right_blobs else None

        if left_boundary and right_boundary:
            lx, ly = left_boundary[0],  left_boundary[1]
            rx, ry = right_boundary[0], right_boundary[1]
            mx, my = (lx + rx) // 2, (ly + ry) // 2
            err = mx - center_x
            cv2.line(vis, (lx, ly), (rx, ry), (0, 255, 0), 2)
            cv2.circle(vis, (mx, my), 12, (0, 255, 0), -1)
            cv2.arrowedLine(vis, bot_pt, (mx, my), (0, 255, 0), 3, tipLength=0.05)
            cv2.putText(vis, f"mid={mx} err={err:+d} L={len(left_blobs)} R={len(right_blobs)}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        else:
            missing = []
            if not left_boundary:  missing.append('NO LEFT')
            if not right_boundary: missing.append('NO RIGHT')
            cv2.putText(vis, ' '.join(missing),
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        combined = np.hstack([vis, mask_bgr])
        image_widget.value = bgr_to_jpeg(combined)
        time.sleep(0.03)

except KeyboardInterrupt:
    print("Preview stopped.")

In [ ]:
cap.release()
print("Camera released.")